# Petroleum Supply Analysis — Phase 2
## 1. Verify Raw Data Access
Check that the historical and incremental raw files are available.

In [0]:
from pathlib import Path

# Location of the validated Phase 1 raw data.
raw_base_path = Path(
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded/"
    "20261003T114452938244Z"
)

if not raw_base_path.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {raw_base_path}")

for load_type in ("full_load", "incremental_load"):
    load_path = raw_base_path / load_type
    json_files = sorted(load_path.glob("page_*.json"))

    if not json_files:
        raise FileNotFoundError(f"No raw JSON files found in: {load_path}")

    total_size_mb = sum(
        file.stat().st_size for file in json_files
    ) / 1_000_000

    print(
        f"{load_type}: "
        f"{len(json_files)} files | "
        f"{total_size_mb:.2f} MB"
    )

print("Raw data access verified.")

## 2. Configure Pipeline Parameters
Select the raw data batch and load type. These parameters allow
historical batches to be reprocessed without changing the pipeline code.

In [0]:
from pathlib import Path

# Default values point to the validated Phase 1 batch.
dbutils.widgets.text(
    "raw_root",
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded",
    "Raw Data Root"
)

dbutils.widgets.text(
    "batch_id",
    "20261003T114452938244Z",
    "Batch ID"
)

dbutils.widgets.dropdown(
    "load_type",
    "full_load",
    ["full_load", "incremental_load"],
    "Load Type"
)

raw_root = dbutils.widgets.get("raw_root").strip()
batch_id = dbutils.widgets.get("batch_id").strip()
load_type = dbutils.widgets.get("load_type").strip()

if not raw_root.startswith("/Volumes/"):
    raise ValueError("Raw Data Root must be a /Volumes/ path.")

if not batch_id or "/" in batch_id or batch_id in {".", ".."}:
    raise ValueError("Batch ID must be a single directory name.")

if load_type not in {"full_load", "incremental_load"}:
    raise ValueError("Unsupported load type.")

source_path = Path(raw_root) / batch_id / load_type
source_files = sorted(source_path.glob("page_*.json"))

if not source_files:
    raise FileNotFoundError(f"No raw JSON files found in: {source_path}")

print(f"Batch ID: {batch_id}")
print(f"Load type: {load_type}")
print(f"Source path: {source_path}")
print(f"Files available: {len(source_files)}")

## 3. Inspect the Raw JSON Structure
Inspect one raw file and one observation before defining the explicit Bronze schema.

In [0]:
import json

# Inspect one file from the selected batch.
sample_file = source_files[0]

with sample_file.open("r", encoding="utf-8") as file:
    payload = json.load(file)

print(f"Sample file: {sample_file.name}")
print(f"Top-level fields: {list(payload.keys())}")

# Support a response object or an API response wrapper.
response = payload.get("response", payload)
records = response.get("data", [])

if not records:
    raise ValueError("The selected file contains no data records.")

print(f"Records in this file: {len(records)}")
print("\nFirst observation:")
print(json.dumps(records[0], indent=2))

print("\nField names and JSON value types:")
for field_name, value in records[0].items():
    print(f"{field_name}: {type(value).__name__}")

## 4. Define the Raw Data Schema
Define the expected observation fields explicitly. Preserve raw values
as strings before validation and conversion in the Silver layer.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Preserve source field names and raw values.
observation_schema = StructType([
    StructField("period", StringType(), True),
    StructField("duoarea", StringType(), True),
    StructField("area-name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("product-name", StringType(), True),
    StructField("process", StringType(), True),
    StructField("process-name", StringType(), True),
    StructField("series", StringType(), True),
    StructField("series-description", StringType(), True),
    StructField("value", StringType(), True),
    StructField("units", StringType(), True),
])

print("Observation schema defined successfully.")
for field in observation_schema.fields:
    print(f"{field.name}: {field.dataType.simpleString()}")

## 5. Test Reading a Raw File with the Explicit Schema
Read one JSON file and expand its data array into individual observations.

In [0]:
from pyspark.sql.types import ArrayType
from pyspark.sql import functions as F

# Observations are stored inside the top-level "data" array.
raw_file_schema = StructType([
    StructField("data", ArrayType(observation_schema), True),
])

# Test the explicit schema on one file.
sample_raw_df = (
    spark.read
    .schema(raw_file_schema)
    .option("multiLine", "true")
    .json(str(source_files[0]))
)

# Convert each observation in the array into a separate row.
sample_observations_df = (
    sample_raw_df
    .select(F.explode("data").alias("observation"))
    .select("observation.*")
)

print(f"Sample observation count: {sample_observations_df.count()}")
display(sample_observations_df.limit(5))

## 6. Read the Selected Raw Batch
Read all JSON pages in the selected batch using the explicit schema
and count the extracted observations.

In [0]:
# Read every JSON page from the parameter-selected folder.
raw_batch_df = (
    spark.read
    .schema(raw_file_schema)
    .option("multiLine", "true")
    .json([str(file_path) for file_path in source_files])
)

# Expand the nested observations into rows.
raw_observations_df = (
    raw_batch_df
    .select(F.explode("data").alias("observation"))
    .select("observation.*")
)

observation_count = raw_observations_df.count()

print(f"Batch ID: {batch_id}")
print(f"Load type: {load_type}")
print(f"Input files: {len(source_files)}")
print(f"Observations read: {observation_count:,}")

## 7. Validate Observation Keys
Check for missing observation keys and duplicate period-series
combinations within the selected batch.

In [0]:
# Each monthly observation is identified by period and series.
key_columns = ["period", "series"]

missing_key_condition = (
    F.col("period").isNull()
    | (F.trim(F.col("period")) == "")
    | F.col("series").isNull()
    | (F.trim(F.col("series")) == "")
)

missing_key_count = (
    raw_observations_df
    .filter(missing_key_condition)
    .count()
)

# Count repeated keys without removing any source records.
duplicate_keys_df = (
    raw_observations_df
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys_df.count()

print(f"Records with missing keys: {missing_key_count:,}")
print(f"Duplicate key combinations: {duplicate_key_count:,}")

if missing_key_count == 0 and duplicate_key_count == 0:
    print("Observation key checks passed.")
else:
    print("Key issues detected. Review before proceeding.")
    display(duplicate_keys_df.limit(10))

## 8. Configure Pipeline Tables
Define the target tables for Bronze, Silver, quarantined records,
and pipeline execution logs.

In [0]:
# Keep all project tables in the existing workspace.default schema.
target_catalog = "workspace"
target_schema = "default"
table_prefix = f"{target_catalog}.{target_schema}"

bronze_table = f"{table_prefix}.petroleum_bronze"
silver_table = f"{table_prefix}.petroleum_silver"
quarantine_table = f"{table_prefix}.petroleum_quarantine"
execution_log_table = f"{table_prefix}.petroleum_execution_logs"

print(f"Bronze table: {bronze_table}")
print(f"Silver table: {silver_table}")
print(f"Quarantine table: {quarantine_table}")
print(f"Execution log table: {execution_log_table}")

## 9. Create the Execution Log Table
Create a Delta table to record processing results, execution times,
row counts, and errors for each pipeline operation.

In [0]:
# Use UTC consistently for pipeline timestamps.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Preserve existing logs when this setup cell is rerun.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {execution_log_table} (
        log_id STRING,
        run_id STRING,
        batch_id STRING,
        load_type STRING,
        layer STRING,
        source_reference STRING,
        target_table STRING,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        status STRING,
        rows_read BIGINT,
        rows_inserted BIGINT,
        rows_updated BIGINT,
        rows_quarantined BIGINT,
        error_message STRING,
        load_timestamp TIMESTAMP
    )
    USING DELTA
""")

print(f"Execution log table ready: {execution_log_table}")

## 10. Define the Execution Logging Function
Create a reusable function to save processing results for each
file or table operation.

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

# Reuse the explicitly defined schema of the execution log table.
execution_log_schema = spark.table(execution_log_table).schema


def write_execution_log(
    *,
    run_id,
    batch_id,
    load_type,
    layer,
    source_reference,
    target_table,
    start_time,
    end_time,
    status,
    rows_read=None,
    rows_inserted=None,
    rows_updated=None,
    rows_quarantined=None,
    error_message=None,
):
    """Append one operation result to the execution log table."""
    if status not in {"Success", "Failure"}:
        raise ValueError("Status must be Success or Failure.")

    log_record = {
        "log_id": str(uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "load_type": load_type,
        "layer": layer,
        "source_reference": str(source_reference),
        "target_table": target_table,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": rows_quarantined,
        "error_message": error_message,
        "load_timestamp": datetime.now(timezone.utc),
    }

    log_df = spark.createDataFrame(
        [log_record],
        schema=execution_log_schema,
    )

    log_df.write.format("delta").mode("append").saveAsTable(
        execution_log_table
    )


print("Execution logging function ready.")

## 11. Verify Execution Logging
Count observations in one sample file and record the validation
operation in the execution log table.

In [0]:
# Identify and time this validation operation.
validation_run_id = str(uuid4())
validation_start = datetime.now(timezone.utc)

validation_df = (
    spark.read
    .schema(raw_file_schema)
    .option("multiLine", "true")
    .json(str(source_files[0]))
    .select(F.explode("data").alias("observation"))
)

validation_count = validation_df.count()
validation_end = datetime.now(timezone.utc)

# This operation reads data without inserting into Bronze or Silver.
write_execution_log(
    run_id=validation_run_id,
    batch_id=batch_id,
    load_type=load_type,
    layer="Raw-validation",
    source_reference=source_files[0],
    target_table=None,
    start_time=validation_start,
    end_time=validation_end,
    status="Success",
    rows_read=validation_count,
    rows_inserted=0,
    rows_updated=0,
    rows_quarantined=0,
)

# Retrieve only the log entry for this validation run.
display(
    spark.table(execution_log_table)
    .filter(F.col("run_id") == validation_run_id)
    .select(
        "layer",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "start_time",
        "end_time",
    )
)